# 层类

上一章，我们用张量统一包装了所有数据。但推理函数 `forward()` 依然是一个独立的函数，权重和偏置也是散落在外的变量。网络层数一多，管理这些分散的变量和函数就会变得非常混乱。

**层**，是深度神经网络的一个重要逻辑概念。每一层包含若干人工神经元，每个神经元有自己的参数（权重和偏置）和计算数据的能力。把神经元的参数和计算逻辑打包在一起，就是**层**（Layer）类的核心设计思想。

层将成为**前向传播链路**的主要载体。

In [1]:
from abc import ABC, abstractmethod

import numpy as np

## 基础架构

### 张量

我们为张量添加了一个新属性：特征维度 `shape`，用来获取数组数据的形状。

例如：`Tensor([28.1, 58.0])` 的 `shape` 是 `(1, 2)`。

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础层

**基础层**（Layer）是一个**抽象类**，为所有层定义统一的接口。

``💡 抽象类（Abstract Base Class，ABC）是一种只能被继承、不能直接实例化的类。它的作用是定义接口规范：继承它的子类必须实现所有标记为 @abstractmethod 的方法，否则 Python 会在创建对象时报错。这保证了所有层都有一致的调用方式。``

基础层定义了一个关键成员：推理函数 `forward()`。这是一个抽象方法，每个具体的层必须实现它，描述数据如何在这一层被加工和传递。所有层通过这个接口串联，构成前向传播链路。

```{figure} images/layer.png
:align: center
:width: 580px
**图例：层的前向传播链路**

* $x$：（输入）特征值；
* $h$：中间值；
* $p$：（输出）预测值。

```

In [3]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    def __repr__(self):
        return f"{type(self).__name__}[]"

## 数据

### 特征

In [4]:
feature = Tensor([28.1, 58.0])

## 模型

### 线性层

我们定义的第一个具体层类是**线性层**（Linear），它封装了线性变换的参数和计算逻辑，等同于一组并行的神经元。

权重矩阵的**每一行**代表一个神经元的权重向量。所以一个有 `out_size` 个神经元（可以产生 `out_size` 个输出值）、接收 `in_size` 个输入特征的线性层，权重矩阵的形状是 `(out_size, in_size)`。

创建线性层需要指定两个参数：

* **输入特征维度 `in_size`**：每个输入样本有几个特征值。我们的例子里是 2（温度、湿度）；
* **输出特征维度 `out_size`**：这一层有几个神经元，也就是输出几个值。我们的例子里是 1（预测销量）。

```{figure} images/layer-weight.png
:align: center
:width: 400px
**图例：含四个神经元的线性层结构**

* $x_1、x_2$：（输入）特征值；
* $w_{n,1}、w_{n,2}、b_n$：第 $n$ 个神经元的权重和偏置；
* $p_1、p_2、p_3、p_4$：（输出）预测值。

```

In [5]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        return Tensor(x.data @ self.weight.data.T + self.bias.data)

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

## 验证

### 建模

创建一个只有一个神经元、接收两个特征值的线性层。

In [6]:
layer = Linear(2, 1)
print(layer)

Linear[weight(1, 2); bias(1,)]


### 推理

调用 `layer()` 时，Python 会触发 `__call__`，进而调用 `forward()`，完成线性计算。

In [7]:
prediction = layer(feature)
print(f'prediction: {prediction}')

prediction: Tensor([43.05])


结果依然是 `43.05`，与上一章的独立函数相同。区别在于，现在权重和偏置不再是散落在外的变量，而是被封装在 `layer` 对象内部，由层类统一管理。

有了层的概念，前向传播就变成了数据在一个个层对象中依次流动的过程。

## 课后练习

尝试增加一个输入特征：风速。你需要修改哪些地方？